# 03 — Hard Negative Mining cho Dense Retriever

Notebook chạy trên Google Colab A100 và tạo dữ liệu theo chiến lược:

```text
1 query + 1 positive + nhiều hard negatives
```

Ứng viên được lấy từ BM25 và Dense Retriever thắng benchmark, hợp nhất bằng RRF, sau đó lọc false negatives và xuất group/pair/triplet.

In [ ]:
# %%capture
# !pip -q install -U pyarrow  torch transformers sentence-transformers accelerate bm25s underthesea tqdm

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%%capture
!pip install bm25s, underthesea

In [3]:
%%capture
!pip install bm25s
!pip install underthesea
from pathlib import Path
from dataclasses import dataclass, asdict
import gc, json, math, random, re, unicodedata
import bm25s
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from underthesea import word_tokenize

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32=True

PROCESSED_DIR=Path('/content/drive/MyDrive/UIT_Legal_System/Dataset/processed')
BENCHMARK_DIR=Path('/content/drive/MyDrive/UIT_Legal_System/Dataset/artifacts/embedding_benchmark')
OUTPUT_DIR=Path('/content/drive/MyDrive/UIT_Legal_System/Dataset/artifacts/hard_negative_mining')
CACHE_DIR=OUTPUT_DIR/'cache'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True); CACHE_DIR.mkdir(parents=True, exist_ok=True)

BM25_TOP_K=100
DENSE_TOP_K=100
RRF_K=60
MIN_NEGATIVES_PER_QUERY=4
TARGET_NEGATIVES_PER_QUERY=12
MAX_NEGATIVES_PER_QUERY=15
MAX_POSITIVE_SIMILARITY_FOR_NEGATIVE=0.92
MIN_QUERY_SIMILARITY_FOR_HARD_NEGATIVE=0.20
MAX_ANSWER_TOKEN_OVERLAP=0.60
EXCLUDE_SAME_ARTICLE=True
EXCLUDE_SAME_DOCUMENT=False
EXCLUDE_EXACT_CONTENT_DUPLICATE=True
USE_CACHE=True
FORCE_RECOMPUTE=False
TORCH_DTYPE=torch.bfloat16
QWEN_TASK_INSTRUCTION=('Given a Vietnamese question about school regulations, retrieve the most relevant regulation passage that provides the factual basis needed to answer the question.')

print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [4]:
def norm(x):
    if pd.isna(x): return ''
    x=unicodedata.normalize('NFKC',str(x)).replace('\u00a0',' ')
    return re.sub(r'\s+',' ',x).strip()
def key(x): return norm(x).lower()
def tokset(x): return {t for t in re.findall(r'\w+',key(x),flags=re.UNICODE) if len(t)>1}
def overlap(a,b):
    A,B=tokset(a),tokset(b)
    return len(A&B)/len(A) if A and B else 0.0
def rrf(dr,br):
    return (0 if dr is None else 1/(RRF_K+dr))+(0 if br is None else 1/(RRF_K+br))
def clear_mem():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

In [5]:
corpus=pd.read_parquet(PROCESSED_DIR/'corpus.parquet').reset_index(drop=True)
train=pd.read_parquet(PROCESSED_DIR/'train_retriever.parquet').reset_index(drop=True)
with open(BENCHMARK_DIR/'zero_shot_winner_config.json',encoding='utf-8') as f: winner=json.load(f)
required_c={'passage_id','content','embedding_text','document','article','content_hash'}
required_t={'qa_id','passage_id','question','document','article','extractive answer','abstractive answer'}
assert required_c<=set(corpus.columns), required_c-set(corpus.columns)
assert required_t<=set(train.columns), required_t-set(train.columns)
passage_ids=corpus.passage_id.astype(str).tolist(); pid2idx={p:i for i,p in enumerate(passage_ids)}
assert train.passage_id.isin(pid2idx).all()
print(winner); print('corpus',len(corpus),'train',len(train))

{'selected_on_split': 'strict_val', 'model_key': 'bge_m3', 'model_name': 'BAAI/bge-m3', 'corpus_field': 'embedding_text', 'query_max_length': 128, 'passage_max_length': 512, 'embedding_dim': 1024, 'metrics': {'Recall@1': 0.2240085744908896, 'Recall@3': 0.47266881028938906, 'Recall@5': 0.6184351554126474, 'Recall@10': 0.752411575562701, 'Recall@20': 0.8231511254019293, 'MRR@10': 0.38565431531669475, 'nDCG@10': 0.4734614605363139, 'MAP@10': 0.38565431531669475}, 'qwen_task_instruction': None, 'note': 'This file records the zero-shot benchmark winner. The next notebook may still mine hard negatives and compare fine-tuned checkpoints against this baseline.'}
corpus 1045 train 7718


In [6]:
FORMATS={
'bge_m3':('raw','raw',False),
'multilingual_e5_base':('e5q','e5p',False),
'bkai_vietnamese_bi_encoder':('raw','raw',True),
}
model_key=winner['model_key']; qfmt,pfmt,segment=FORMATS[model_key]
model_name=winner['model_name']; corpus_field=winner['corpus_field']
qmax=int(winner.get('query_max_length',128)); pmax=int(winner.get('passage_max_length',512))
def fmt(texts,mode):
    if mode=='qwen': out=[f'Instruct: {QWEN_TASK_INSTRUCTION}\nQuery:{x}' for x in texts]
    elif mode=='e5q': out=[f'query: {x}' for x in texts]
    elif mode=='e5p': out=[f'passage: {x}' for x in texts]
    else: out=list(texts)
    return [word_tokenize(x,format='text') for x in out] if segment else out

In [7]:
# BM25
bm25_dir=OUTPUT_DIR/'bm25_index'; bm25_cache=CACHE_DIR/'train_bm25_topk.npz'
passage_texts=corpus[corpus_field].fillna('').astype(str).tolist(); questions=train.question.fillna('').astype(str).tolist()
if bm25_dir.exists() and USE_CACHE and not FORCE_RECOMPUTE:
    bm25=bm25s.BM25.load(str(bm25_dir),load_corpus=False)
else:
    ct=bm25s.tokenize(passage_texts,stopwords=None,stemmer=None,show_progress=True)
    bm25=bm25s.BM25(method='lucene'); bm25.index(ct,show_progress=True); bm25.save(str(bm25_dir),corpus=None)
if bm25_cache.exists() and USE_CACHE and not FORCE_RECOMPUTE:
    z=np.load(bm25_cache); bm25_idx,bm25_score=z['indices'],z['scores']
else:
    qt=bm25s.tokenize(questions,stopwords=None,stemmer=None,show_progress=True)
    bm25_idx,bm25_score=bm25.retrieve(qt,k=min(BM25_TOP_K,len(corpus)),show_progress=True)
    np.savez_compressed(bm25_cache,indices=bm25_idx,scores=bm25_score)
print(bm25_idx.shape)

(7718, 100)


In [8]:
# Dense model + embeddings
clear_mem(); device='cuda' if torch.cuda.is_available() else 'cpu'
kwargs={'device':device,'model_kwargs':{'torch_dtype':TORCH_DTYPE}}
if model_key=='qwen3_embedding_4b':
    try:
        model=SentenceTransformer(model_name,device=device,model_kwargs={'torch_dtype':TORCH_DTYPE,'attn_implementation':'flash_attention_2'},tokenizer_kwargs={'padding_side':'left'})
    except Exception:
        clear_mem(); model=SentenceTransformer(model_name,device=device,model_kwargs={'torch_dtype':TORCH_DTYPE,'attn_implementation':'sdpa'},tokenizer_kwargs={'padding_side':'left'})
else: model=SentenceTransformer(model_name,**kwargs)
cp=CACHE_DIR/f'{model_key}_{corpus_field}_corpus.npy'; qp=CACHE_DIR/f'{model_key}_train_queries.npy'
if cp.exists() and USE_CACHE and not FORCE_RECOMPUTE: C=np.load(cp)
else:
    model.max_seq_length=pmax; C=model.encode(fmt(passage_texts,pfmt),batch_size=8 if 'qwen' in model_key else 16,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True).astype('float32'); np.save(cp,C)
if qp.exists() and USE_CACHE and not FORCE_RECOMPUTE: Q=np.load(qp)
else:
    model.max_seq_length=qmax; Q=model.encode(fmt(questions,qfmt),batch_size=16 if 'qwen' in model_key else 32,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True).astype('float32'); np.save(qp,Q)
print(C.shape,Q.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/66 [00:00<?, ?it/s]

Batches:   0%|          | 0/242 [00:00<?, ?it/s]

(1045, 1024) (7718, 1024)


In [9]:
# Dense top-k
dense_cache=CACHE_DIR/'train_dense_topk.npz'
def topk(Q,C,k,chunk=256):
    dev='cuda' if torch.cuda.is_available() else 'cpu'; Ct=torch.from_numpy(C).to(dev); II=[]; SS=[]
    for s in tqdm(range(0,len(Q),chunk),desc='dense retrieval'):
        qt=torch.from_numpy(Q[s:s+chunk]).to(dev); sim=qt@Ct.T; sc,ix=torch.topk(sim,k=min(k,len(C)),dim=1)
        II.append(ix.cpu().numpy()); SS.append(sc.float().cpu().numpy()); del qt,sim,sc,ix
    del Ct; clear_mem(); return np.concatenate(II),np.concatenate(SS)
if dense_cache.exists() and USE_CACHE and not FORCE_RECOMPUTE:
    z=np.load(dense_cache); dense_idx,dense_score=z['indices'],z['scores']
else:
    dense_idx,dense_score=topk(Q,C,DENSE_TOP_K); np.savez_compressed(dense_cache,indices=dense_idx,scores=dense_score)
print(dense_idx.shape)

(7718, 100)


In [10]:
# Candidate generation
rows=[]
for qi,tr in tqdm(train.iterrows(),total=len(train),desc='candidates'):
    pos_id=str(tr.passage_id); pos_i=pid2idx[pos_id]; cand={}
    for rank,ci in enumerate(dense_idx[qi],1):
        ci=int(ci); pid=passage_ids[ci]; cand.setdefault(pid,{'ci':ci,'dr':None,'ds':None,'br':None,'bs':None}); cand[pid].update(dr=rank,ds=float(dense_score[qi,rank-1]))
    for rank,ci in enumerate(bm25_idx[qi],1):
        ci=int(ci); pid=passage_ids[ci]; cand.setdefault(pid,{'ci':ci,'dr':None,'ds':None,'br':None,'bs':None}); cand[pid].update(br=rank,bs=float(bm25_score[qi,rank-1]))
    ans=norm(tr['abstractive answer']) or norm(tr['extractive answer']); pos=corpus.iloc[pos_i]
    for pid,x in cand.items():
        cr=corpus.iloc[x['ci']]
        rows.append({'qa_id':tr.qa_id,'question':tr.question,'positive_passage_id':pos_id,'candidate_passage_id':pid,'candidate_corpus_index':x['ci'],'dense_rank':x['dr'],'dense_score':x['ds'],'bm25_rank':x['br'],'bm25_score':x['bs'],'rrf_score':rrf(x['dr'],x['br']),'query_similarity':float(Q[qi]@C[x['ci']]),'positive_similarity':float(C[pos_i]@C[x['ci']]),'same_passage':pid==pos_id,'same_document':key(cr.document)==key(tr.document),'same_article':key(cr.article)==key(tr.article),'same_content_hash':cr.content_hash==pos.content_hash,'answer_token_overlap':overlap(ans,cr.content),'candidate_document':cr.document,'candidate_article':cr.article,'candidate_content':cr.content,'candidate_text':cr[corpus_field]})
cand=pd.DataFrame(rows); cand.to_parquet(OUTPUT_DIR/'hard_negative_candidates_raw.parquet',index=False)
print(len(cand))

candidates:   0%|          | 0/7718 [00:00<?, ?it/s]

1191220


In [11]:
# False-negative filtering + hardness
def reason(r):
    if r.same_passage: return 'positive_passage'
    if EXCLUDE_EXACT_CONTENT_DUPLICATE and r.same_content_hash: return 'duplicate_content'
    if EXCLUDE_SAME_ARTICLE and r.same_article and r.same_document: return 'same_article'
    if EXCLUDE_SAME_DOCUMENT and r.same_document: return 'same_document'
    if r.positive_similarity>MAX_POSITIVE_SIMILARITY_FOR_NEGATIVE: return 'too_similar_to_positive'
    if r.answer_token_overlap>MAX_ANSWER_TOKEN_OVERLAP: return 'high_answer_overlap'
    if r.query_similarity<MIN_QUERY_SIMILARITY_FOR_HARD_NEGATIVE: return 'too_easy'
    return None
cand['filter_reason']=cand.apply(reason,axis=1); valid=cand[cand.filter_reason.isna()].copy()
valid['bm25_norm']=valid.groupby('qa_id').bm25_score.transform(lambda s:(s-s.min())/(s.max()-s.min()+1e-8)).fillna(0)
inv=lambda x:0 if pd.isna(x) else 1/float(x)
valid['hardness_score']=0.35*valid.query_similarity+25*valid.rrf_score+0.15*valid.dense_rank.map(inv)+0.15*valid.bm25_rank.map(inv)+0.10*valid.bm25_norm
valid['retrieval_source']=np.select([valid.dense_rank.notna()&valid.bm25_rank.notna(),valid.dense_rank.notna(),valid.bm25_rank.notna()],['dense+bm25','dense','bm25'],'unknown')
valid.sort_values(['qa_id','hardness_score'],ascending=[True,False],inplace=True)
valid.to_parquet(OUTPUT_DIR/'hard_negative_candidates.parquet',index=False)
rep=cand.filter_reason.fillna('valid_negative').value_counts().rename_axis('reason').reset_index(name='count'); rep['ratio']=rep['count']/len(cand); rep.to_csv(OUTPUT_DIR/'filter_report.csv',index=False,encoding='utf-8-sig'); display(rep)

,reason,count,ratio
0,valid_negative,735173,0.617160
1,high_answer_overlap,417632,0.350592
2,duplicate_content,29593,0.024843
3,positive_passage,7696,0.006461
4,too_similar_to_positive,686,0.000576
5,too_easy,269,0.000226
6,same_article,171,0.000144


In [12]:
# Diversified selection
def select(g):
    chosen=[]; seen=set()
    def add(mask,n):
        for idx,r in g[mask & ~g.candidate_passage_id.isin(seen)].sort_values('hardness_score',ascending=False).head(n).iterrows(): chosen.append(idx); seen.add(r.candidate_passage_id)
    add(g.bm25_rank.notna(),3); add(g.dense_rank.notna(),3); add(g.retrieval_source.eq('dense+bm25'),2)
    add(pd.Series(True,index=g.index),TARGET_NEGATIVES_PER_QUERY-len(chosen))
    return g.loc[chosen].drop_duplicates('candidate_passage_id').sort_values('hardness_score',ascending=False).head(MAX_NEGATIVES_PER_QUERY)
selected={}
for q,g in tqdm(valid.groupby('qa_id'),total=valid.qa_id.nunique(),desc='select'):
    s=select(g)
    if len(s)>=MIN_NEGATIVES_PER_QUERY: selected[q]=s
print(len(selected),'/',train.qa_id.nunique())

select:   0%|          | 0/7718 [00:00<?, ?it/s]

7718 / 7718


In [13]:
# Export group, pair, triplet
train_by=train.set_index('qa_id'); groups=[]; pairs=[]; trips=[]; selected_rows=[]
for q,ng in tqdm(selected.items(),desc='export'):
    tr=train_by.loc[q]; pos_id=str(tr.passage_id); pos=corpus.iloc[pid2idx[pos_id]]; negs=[]
    for order,(_,r) in enumerate(ng.iterrows(),1):
        rec={'passage_id':r.candidate_passage_id,'text':r.candidate_text,'content':r.candidate_content,'document':r.candidate_document,'article':r.candidate_article,'dense_rank':None if pd.isna(r.dense_rank) else int(r.dense_rank),'bm25_rank':None if pd.isna(r.bm25_rank) else int(r.bm25_rank),'dense_score':None if pd.isna(r.dense_score) else float(r.dense_score),'bm25_score':None if pd.isna(r.bm25_score) else float(r.bm25_score),'rrf_score':float(r.rrf_score),'query_similarity':float(r.query_similarity),'positive_similarity':float(r.positive_similarity),'hardness_score':float(r.hardness_score),'retrieval_source':r.retrieval_source}
        negs.append(rec); trips.append({'qa_id':q,'query':tr.question,'positive':pos[corpus_field],'negative':r.candidate_text,'positive_passage_id':pos_id,'negative_passage_id':r.candidate_passage_id,'negative_order':order,'hardness_score':float(r.hardness_score),'retrieval_source':r.retrieval_source}); selected_rows.append(r.to_dict())
    groups.append({'qa_id':q,'query':tr.question,'positive':pos[corpus_field],'positive_passage_id':pos_id,'positive_document':pos.document,'positive_article':pos.article,'hard_negatives':negs,'num_hard_negatives':len(negs),'model_key':model_key,'corpus_field':corpus_field}); pairs.append({'qa_id':q,'query':tr.question,'positive':pos[corpus_field],'positive_passage_id':pos_id})
groups_df=pd.DataFrame(groups); pairs_df=pd.DataFrame(pairs); trips_df=pd.DataFrame(trips); sel_df=pd.DataFrame(selected_rows)
groups_df.to_parquet(OUTPUT_DIR/'train_groups.parquet',index=False); pairs_df.to_parquet(OUTPUT_DIR/'train_pairs.parquet',index=False); trips_df.to_parquet(OUTPUT_DIR/'train_triplets.parquet',index=False); sel_df.to_parquet(OUTPUT_DIR/'selected_hard_negatives.parquet',index=False)
with open(OUTPUT_DIR/'train_groups.jsonl','w',encoding='utf-8') as f:
    for r in groups: f.write(json.dumps(r,ensure_ascii=False)+'\n')
print(len(groups_df),len(trips_df))

export:   0%|          | 0/7718 [00:00<?, ?it/s]

7718 92610


In [14]:
# Reports and config
stats={'num_train_queries':int(train.qa_id.nunique()),'num_queries_with_negatives':int(len(groups_df)),'coverage_ratio':float(len(groups_df)/max(train.qa_id.nunique(),1)),'avg_negatives_per_query':float(groups_df.num_hard_negatives.mean()),'min_negatives_per_query':int(groups_df.num_hard_negatives.min()),'max_negatives_per_query':int(groups_df.num_hard_negatives.max()),'avg_query_similarity':float(sel_df.query_similarity.mean()),'avg_positive_similarity':float(sel_df.positive_similarity.mean()),'same_document_ratio':float(sel_df.same_document.mean()),'source_distribution':sel_df.retrieval_source.value_counts(normalize=True).to_dict()}
config={'schema_version':'1.0','strategy':'one query + one positive + multiple hard negatives','winner':winner,'bm25_top_k':BM25_TOP_K,'dense_top_k':DENSE_TOP_K,'rrf_k':RRF_K,'target_negatives':TARGET_NEGATIVES_PER_QUERY,'filters':{'max_positive_similarity':MAX_POSITIVE_SIMILARITY_FOR_NEGATIVE,'min_query_similarity':MIN_QUERY_SIMILARITY_FOR_HARD_NEGATIVE,'max_answer_overlap':MAX_ANSWER_TOKEN_OVERLAP,'exclude_same_article':EXCLUDE_SAME_ARTICLE,'exclude_same_document':EXCLUDE_SAME_DOCUMENT}}
for name,obj in [('mining_statistics.json',stats),('mining_config.json',config)]:
    with open(OUTPUT_DIR/name,'w',encoding='utf-8') as f: json.dump(obj,f,ensure_ascii=False,indent=2)
sel_df.groupby('qa_id').agg(num_negatives=('candidate_passage_id','count'),avg_hardness=('hardness_score','mean'),avg_query_similarity=('query_similarity','mean'),same_document_ratio=('same_document','mean')).reset_index().to_csv(OUTPUT_DIR/'per_query_mining_report.csv',index=False,encoding='utf-8-sig')
sel_df.sort_values(['qa_id','hardness_score'],ascending=[True,False]).groupby('qa_id').head(3).head(60).to_csv(OUTPUT_DIR/'manual_inspection_sample.csv',index=False,encoding='utf-8-sig')
print(json.dumps(stats,ensure_ascii=False,indent=2))

{
  "num_train_queries": 7718,
  "num_queries_with_negatives": 7718,
  "coverage_ratio": 1.0,
  "avg_negatives_per_query": 11.999222596527598,
  "min_negatives_per_query": 9,
  "max_negatives_per_query": 12,
  "avg_query_similarity": 0.5621543960078413,
  "avg_positive_similarity": 0.716677712386595,
  "same_document_ratio": 0.23443472627146097,
  "source_distribution": {
    "dense+bm25": 0.8221898283122773,
    "dense": 0.10850880034553503,
    "bm25": 0.06930137134218767
  }
}


In [15]:
# Final validation
required=['train_groups.parquet','train_groups.jsonl','train_pairs.parquet','train_triplets.parquet','hard_negative_candidates.parquet','selected_hard_negatives.parquet','filter_report.csv','per_query_mining_report.csv','manual_inspection_sample.csv','mining_statistics.json','mining_config.json']
missing=[x for x in required if not (OUTPUT_DIR/x).exists()]
if missing: raise RuntimeError(missing)
if len(groups_df)==0 or groups_df.num_hard_negatives.min()<MIN_NEGATIVES_PER_QUERY: raise RuntimeError('Invalid groups')
print('Hard-negative mining completed successfully.')
for x in sorted(OUTPUT_DIR.glob('*')):
    if x.is_file(): print(f'{x.name:44s}{x.stat().st_size/1024:12.2f} KB')

Hard-negative mining completed successfully.
filter_report.csv                                   0.32 KB
hard_negative_candidates.parquet               329375.22 KB
hard_negative_candidates_raw.parquet           613151.46 KB
manual_inspection_sample.csv                      257.96 KB
mining_config.json                                  1.12 KB
mining_statistics.json                              0.47 KB
per_query_mining_report.csv                       595.11 KB
selected_hard_negatives.parquet                 37801.50 KB
train_groups.jsonl                             375224.29 KB
train_groups.parquet                            36944.57 KB
train_pairs.parquet                               721.08 KB
train_triplets.parquet                          33820.65 KB


## Output chính

```text
artifacts/hard_negative_mining/
├── train_groups.parquet
├── train_groups.jsonl
├── train_pairs.parquet
├── train_triplets.parquet
├── hard_negative_candidates.parquet
├── selected_hard_negatives.parquet
├── filter_report.csv
├── per_query_mining_report.csv
├── manual_inspection_sample.csv
├── mining_statistics.json
└── mining_config.json
```

Notebook 04 sẽ ưu tiên `train_groups.parquet`, trong đó mỗi dòng chứa query, positive và danh sách hard negatives.